# Detection rate vs source position

Requires a run saved with ``--save-source``.

1. Detection-fraction histogram (linear + log y).
2. Scatter vs ρ / z with rolling-median overlay.
3. 2D ``(ρ, z)`` binned heatmap (linear + log color).
4. Detection vs distance-to-nearest-wall.
5. 3D source-position scatter coloured by detection fraction.


In [ ]:
import sys
sys.path.append('../../../../')  # notebooks/ → photon_shotgun/ → production/ → lucid/ → repo root

import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from lucid.production.photon_shotgun.io import (
    load_shotgun_waveform, load_shotgun_per_photon,
)
from lucid.production.photon_shotgun.viz import (
    scatter_sensors_3d, plot_hist_lin_log,
)
from lucid.geometry import generate_detector

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 5)


def _resolve_geom(meta):
    geom_path = meta.get('detector_config', '')
    if isinstance(geom_path, bytes):
        geom_path = geom_path.decode()
    for candidate in ['../../../../' + geom_path, geom_path]:
        if os.path.exists(candidate):
            return candidate
    return geom_path


In [ ]:
PATH = '../../../../runs/shotgun_SK_1k_waveform.h5'
MODE = 'waveform'   # or 'per_photon'

if MODE == 'waveform':
    out = load_shotgun_waveform(PATH)
    n_detected = out['n_detected']
else:
    out = load_shotgun_per_photon(PATH)
    n_detected = out['detected'].sum(axis=1)

meta = dict(out['meta'])
n_cases = int(meta['n_cases'])
n_photons = int(meta.get('n_photons',
                         out['detected'].shape[1] if MODE == 'per_photon' else 100_000))
frac = n_detected / n_photons
print(f"n_cases={n_cases}, n_photons={n_photons}")
print(f"detection fraction: mean={frac.mean():.4f}  median={np.median(frac):.4f}  "
      f"[{frac.min():.4f}, {frac.max():.4f}]")


In [ ]:
plot_hist_lin_log(frac, bins=60,
                  title=f'{n_cases:,} cases × {n_photons:,} photons',
                  xlabel='detection fraction')
plt.show()


## Source positions


In [ ]:
src = out.get('source')
if src is None:
    raise RuntimeError('Re-run with --save-source to persist per-case '
                       'origin / direction arrays')

pos = np.asarray(src.origins)[:, 0, :]        # (n_cases, 3)
rho = np.hypot(pos[:, 0], pos[:, 1])
z = pos[:, 2]

det = generate_detector(_resolve_geom(meta))
sensor_points = np.asarray(det.all_points)
R = float(np.hypot(sensor_points[:, 0], sensor_points[:, 1]).max())
H = float(getattr(det, 'H'))
print(f"detector: R={R:.2f} m, H={H:.2f} m")
print(f"position extent: ρ ∈ [{rho.min():.2f}, {rho.max():.2f}], "
      f"z ∈ [{z.min():.2f}, {z.max():.2f}]")


## Scatter + rolling median vs ρ and z


In [ ]:
def _rolling_median(x, y, n_bins=40):
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    bins = np.linspace(xs[0], xs[-1], n_bins + 1)
    idx = np.digitize(xs, bins[1:-1])
    med = np.array([np.median(ys[idx == i]) if (idx == i).any() else np.nan
                    for i in range(n_bins)])
    centers = 0.5 * (bins[:-1] + bins[1:])
    return centers, med

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, x_, name in zip(axes, (rho, z), ('ρ', 'z')):
    ax.scatter(x_, frac, s=3, alpha=0.25, color='steelblue')
    cx, mx = _rolling_median(x_, frac)
    ax.plot(cx, mx, 'red', lw=2, label='rolling median')
    ax.set_xlabel(f'{name} (m)')
    ax.set_ylabel('detection fraction')
    ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=8)
fig.suptitle('detection fraction vs source position')
fig.tight_layout()
plt.show()


## 2D ``(ρ, z)`` heatmap — linear + log color


In [ ]:
from matplotlib.colors import LogNorm, Normalize

H_sum, x_edges, y_edges = np.histogram2d(rho, z, bins=[40, 40], weights=n_detected)
H_cnt, _, _ = np.histogram2d(rho, z, bins=[x_edges, y_edges])
with np.errstate(invalid='ignore'):
    mean_frac = (H_sum / np.maximum(H_cnt, 1)) / n_photons

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, scale in zip(axes, ('linear', 'log')):
    data = mean_frac.T
    if scale == 'log':
        positive = data[data > 0]
        norm = LogNorm(vmin=positive.min(), vmax=data.max()) if positive.size else Normalize()
    else:
        norm = Normalize(vmin=0.0, vmax=float(data.max()))
    mesh = ax.pcolormesh(x_edges, y_edges, data, shading='auto',
                         cmap='magma', norm=norm)
    fig.colorbar(mesh, ax=ax, label='mean detection fraction')
    ax.set_xlabel('ρ (m)')
    ax.set_ylabel('z (m)')
    ax.set_title(f'{scale} scale')
fig.suptitle('detection fraction vs (ρ, z)')
fig.tight_layout()
plt.show()


## Detection vs distance-to-nearest-wall


In [ ]:
d_wall = np.minimum(R - rho, H / 2 - np.abs(z))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, yscale in zip(axes, ('linear', 'log')):
    ax.scatter(d_wall, frac, s=3, alpha=0.25, color='coral')
    cx, mx = _rolling_median(d_wall, frac)
    ax.plot(cx, mx, 'black', lw=2, label='rolling median')
    ax.set_xlabel('distance to nearest wall (m)')
    ax.set_ylabel('detection fraction')
    ax.set_yscale(yscale)
    ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=8)
fig.suptitle('detection fraction vs wall distance')
fig.tight_layout()
plt.show()


## 3D source-position scatter


In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=frac, cmap='viridis',
                s=6, alpha=0.7)
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_zlabel('z (m)')
ax.set_title('source positions coloured by detection fraction')
fig.colorbar(sc, shrink=0.6, label='detection fraction')
plt.show()
